# LM Judge Accuracy Evaluation — CLAMBER Dataset

Evaluates whether the LM judge correctly classifies clarifying questions as **EPISTEMIC** or **ALEATORIC**
using 100 entries from the CLAMBER benchmark as ground truth.

## Ground-truth mapping from CLAMBER taxonomy

CLAMBER defines three top-level categories, each with subclasses:

| CLAMBER category | Subclass | Mapped label | Rationale |
|---|---|---|---|
| **FD** — Epistemic Misalignment | `NK` (Unfamiliar entity) | **EPISTEMIC** | Model doesn't recognise the entity — a knowledge gap |
| **FD** — Epistemic Misalignment | `ICL` (Contradiction) | **EPISTEMIC** | Conflicting examples expose a knowledge/reasoning gap |
| **LA** — Linguistic Ambiguity | `polysemy` (Lexical) | **ALEATORIC** | Word has multiple valid meanings — irreducible |
| **LA** — Linguistic Ambiguity | `co-reference` (Semantic) | **ALEATORIC** | Pronoun/reference is ambiguous — context-dependent |
| **MC** — Aleatoric Output | `whom` | **ALEATORIC** | Missing personal/preference details |
| **MC** — Aleatoric Output | `what` | **ALEATORIC** | Missing task-specific details |
| **MC** — Aleatoric Output | `when` | **ALEATORIC** | Missing temporal details |
| **MC** — Aleatoric Output | `where` | **ALEATORIC** | Missing spatial details |

**Input to judge:** `clarifying_question` field (ground-truth CQ from CLAMBER)  
**Target:** 50 epistemic (25 NK + 25 ICL) + 50 aleatoric (mixed MC + LA subclasses)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../").resolve()))

# ── Dataset config — change DATASET to switch experiments ─────────────────
DATASET           = "clamber"
ROOT              = Path("../").resolve()
DATASETS_DIR      = ROOT / "datasets" / DATASET
OUTPUTS_DIR       = ROOT / "outputs"  / DATASET
PROMPTS_DIR       = ROOT / "prompts"  / "medqa"   # judge instruction is shared

CLAMBER_PATH         = DATASETS_DIR / "clamber_benchmark.jsonl"
JUDGE_INSTRUCTION    = PROMPTS_DIR  / "judge_instruction.txt"
OUTPUT_CSV           = OUTPUTS_DIR  / "clamber_judge_eval.csv"

# ── Sampling config ───────────────────────────────────────────────────────
N_PER_CLASS       = 50      # 50 epistemic + 50 aleatoric = 100 total
RANDOM_SEED       = 42

# ── Model / run config ────────────────────────────────────────────────────
MODEL_ID          = "gemini-2.5-flash"
REQUEST_INTERVAL  = 1.0
# ─────────────────────────────────────────────────────────────────────────

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import logging
import random
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.judge import LLMJudge, CSVBatchClassifier, FewShotExample

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

## Load & Sample CLAMBER

In [ ]:
# CLAMBER is double-encoded JSON: each line is a JSON string containing JSON
raw_rows = []
with open(CLAMBER_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            outer = json.loads(line)
            raw_rows.append(json.loads(outer))

df = pd.DataFrame(raw_rows)
print(f"Total CLAMBER rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()
print("Subclass counts:")
print(df["subclass"].value_counts())
print()
print("Category counts:")
print(df["category"].value_counts())

In [ ]:
# ── Ground-truth mapping ──────────────────────────────────────────────────
# FD category: NK + ICL → EPISTEMIC
# LA category: polysemy + co-reference → ALEATORIC
# MC category: whom + what + when + where → ALEATORIC
EPISTEMIC_SUBCLASSES = {"NK", "ICL"}
ALEATORIC_SUBCLASSES = {"polysemy", "co-reference", "whom", "what", "when", "where"}

# Only use rows that have a ground-truth clarifying question
has_cq = df["clarifying_question"].notna() & (df["clarifying_question"].str.strip() != "")

epistemic_pool = df[df["subclass"].isin(EPISTEMIC_SUBCLASSES) & has_cq].copy()
aleatoric_pool = df[df["subclass"].isin(ALEATORIC_SUBCLASSES) & has_cq].copy()

print(f"Epistemic pool (NK + ICL): {len(epistemic_pool)}")
print(epistemic_pool["subclass"].value_counts().to_string())
print()
print(f"Aleatoric pool (MC + LA): {len(aleatoric_pool)}")
print(aleatoric_pool["subclass"].value_counts().to_string())

# ── Balanced sampling: 25 NK + 25 ICL for epistemic ──────────────────────
n_each_epi = N_PER_CLASS // 2   # 25 per epistemic subclass

epi_nk  = epistemic_pool[epistemic_pool["subclass"] == "NK"].sample(n=n_each_epi, random_state=RANDOM_SEED)
epi_icl = epistemic_pool[epistemic_pool["subclass"] == "ICL"].sample(n=n_each_epi, random_state=RANDOM_SEED)
epi_sample = pd.concat([epi_nk, epi_icl], ignore_index=True)

# 50 aleatoric drawn proportionally across subclasses
ale_sample = aleatoric_pool.sample(n=N_PER_CLASS, random_state=RANDOM_SEED)

epi_sample = epi_sample.copy()
ale_sample = ale_sample.copy()
epi_sample["true_label"] = "EPISTEMIC"
ale_sample["true_label"] = "ALEATORIC"

eval_df = pd.concat([epi_sample, ale_sample], ignore_index=True).sample(
    frac=1, random_state=RANDOM_SEED   # shuffle
).reset_index(drop=True)

eval_df["eval_id"] = [f"clamber_{i:03d}" for i in range(len(eval_df))]

print(f"\nEvaluation set: {len(eval_df)} rows")
print("\nSubclass distribution in eval set:")
print(eval_df.groupby(["true_label", "subclass"]).size().to_string())
print()
print(eval_df[["eval_id", "subclass", "true_label", "clarifying_question"]].head(10).to_string(index=False))

## Run LM Judge

In [ ]:
# Save the eval set as the input CSV for the batch classifier
INPUT_CSV = OUTPUTS_DIR / "clamber_eval_input.csv"
eval_df[["eval_id", "subclass", "true_label", "clarifying_question"]].to_csv(
    INPUT_CSV, index=False
)
print(f"Eval input saved to {INPUT_CSV}")

In [ ]:
REGENERATE = False  # set True to re-run classification from scratch

if OUTPUT_CSV.exists() and not REGENERATE:
    print(f"Loading existing results from {OUTPUT_CSV}")
else:
    if REGENERATE and OUTPUT_CSV.exists():
        OUTPUT_CSV.unlink()
        print("Deleted existing output — regenerating.")

    provider = GeminiProvider(model_id=MODEL_ID)

    judge = LLMJudge(
        provider=provider,
        instructions_path=JUDGE_INSTRUCTION,
        label_parser=lambda text: text.strip().upper(),
    )

    classifier = CSVBatchClassifier(
        judge=judge,
        input_csv=INPUT_CSV,
        output_csv=OUTPUT_CSV,
        question_column="clarifying_question",
        id_column="eval_id",
        delay_between_calls=REQUEST_INTERVAL,
    )

    classifier.run()
    print(f"Classification complete. Results at {OUTPUT_CSV}")

## Accuracy Analysis

In [ ]:
results_df = pd.read_csv(OUTPUT_CSV)

# Merge judge output with ground truth
merged = eval_df[["eval_id", "subclass", "true_label", "clarifying_question"]].merge(
    results_df[["id", "label", "latency_seconds", "error"]].rename(columns={"id": "eval_id"}),
    on="eval_id",
    how="left",
)

# Normalise judge labels
merged["predicted_label"] = merged["label"].str.strip().str.upper()
merged["valid_prediction"] = merged["predicted_label"].isin({"EPISTEMIC", "ALEATORIC"})
merged["correct"] = merged["true_label"] == merged["predicted_label"]

n_total   = len(merged)
n_valid   = merged["valid_prediction"].sum()
n_correct = merged["correct"].sum()
accuracy  = merged.loc[merged["valid_prediction"], "correct"].mean()

display(Markdown(f"""
### Overall Results
| Metric | Value |
|---|---|
| Total evaluated | {n_total} |
| Valid predictions | {n_valid} |
| Correct | {n_correct} |
| **Accuracy (valid only)** | **{accuracy:.1%}** |
"""))

print("\nPer-class accuracy:")
per_class = (
    merged[merged["valid_prediction"]]
    .groupby("true_label")["correct"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "correct", "count": "total", "mean": "accuracy"})
)
per_class["accuracy"] = per_class["accuracy"].map("{:.1%}".format)
display(per_class)

print("\nPer-subclass breakdown:")
per_sub = (
    merged[merged["valid_prediction"]]
    .groupby(["subclass", "true_label"])["correct"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "correct", "count": "total", "mean": "accuracy"})
)
per_sub["accuracy"] = per_sub["accuracy"].map("{:.1%}".format)
display(per_sub)

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────
valid = merged[merged["valid_prediction"]]
cm = pd.crosstab(
    valid["true_label"],
    valid["predicted_label"],
    rownames=["True"],
    colnames=["Predicted"],
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=0.5)
ax.set_title(f"LM Judge Confusion Matrix\n(accuracy = {accuracy:.1%})")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUTS_DIR / 'confusion_matrix.png'}")

In [ ]:
# ── Accuracy per subclass bar chart ───────────────────────────────────────
sub_acc = (
    merged[merged["valid_prediction"]]
    .groupby(["subclass", "true_label"])["correct"]
    .mean()
    .reset_index()
    .rename(columns={"correct": "accuracy"})
    .sort_values("accuracy")
)

palette = {"EPISTEMIC": "#1f77b4", "ALEATORIC": "#ff7f0e"}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    sub_acc["subclass"],
    sub_acc["accuracy"],
    color=[palette[l] for l in sub_acc["true_label"]],
    edgecolor="white",
)
ax.axvline(0.5, color="red", linestyle="--", linewidth=0.8, label="Chance (50%)")
ax.set_xlabel("Accuracy")
ax.set_title("Judge Accuracy per CLAMBER Subclass")
ax.set_xlim(0, 1)
# Colour legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in palette.items()]
ax.legend(handles=legend_elements + [plt.Line2D([0],[0], color='red', linestyle='--', label='Chance')],
          loc="lower right")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "accuracy_per_subclass.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Error inspection — where did the judge go wrong? ─────────────────────
errors = merged[merged["valid_prediction"] & ~merged["correct"]].copy()
print(f"Misclassified: {len(errors)} / {n_valid}")

if len(errors) > 0:
    display(Markdown("### Misclassified examples"))
    display(
        errors[["eval_id", "subclass", "true_label", "predicted_label", "clarifying_question"]]
        .head(20)
        .to_string(index=False)
    )

## Reflection

### Overall accuracy — does the judge perform above chance?
_Your notes here._

### Which subclasses are hardest for the judge and why?
_Your notes here._

### NK vs ICL — does the judge handle both epistemic subtypes equally well?
_Your notes here._

### Where does the judge confuse epistemic for aleatoric or vice versa?
_Your notes here._

### What does this tell us about the judge instruction quality?
_Your notes here._